In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

print(os.listdir('/content/drive/MyDrive/data'))

['test_set', 'training_set']


In [4]:
train_path = '/content/drive/MyDrive/data/training_set'
test_path  = '/content/drive/MyDrive/data/test_set'

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "gpu")
print("Device:", device)

Device: cuda


In [6]:
from torchvision import transforms

transform_train = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

transform_test = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5],
                         std=[0.5,0.5,0.5])
])

In [7]:
train_dataset = torchvision.datasets.ImageFolder(
    root=train_path,
    transform=transform_train
)

test_dataset = torchvision.datasets.ImageFolder(
    root=test_path,
    transform=transform_test
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

classes = train_dataset.classes
print(classes)   # ['cats', 'dogs']

['cats', 'dogs']


In [8]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2,2)

        self.fc1 = nn.Linear(256*8*8, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))

        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)

        return x

In [9]:
model = CNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

In [10]:
num_epochs = 20

train_acc_list = []
train_loss_list = []

for epoch in range(num_epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler.step()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    train_loss_list.append(epoch_loss)
    train_acc_list.append(epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} |  Acc: {epoch_acc:.2f}%")

Epoch 1/20 | Loss: 0.8215 |  Acc: 60.82%
Epoch 2/20 | Loss: 0.5889 |  Acc: 68.38%
Epoch 3/20 | Loss: 0.5437 |  Acc: 72.65%
Epoch 4/20 | Loss: 0.5021 |  Acc: 76.43%
Epoch 5/20 | Loss: 0.4761 |  Acc: 78.05%
Epoch 6/20 | Loss: 0.4378 |  Acc: 80.17%
Epoch 7/20 | Loss: 0.4213 |  Acc: 81.40%
Epoch 8/20 | Loss: 0.3928 |  Acc: 83.11%
Epoch 9/20 | Loss: 0.3820 |  Acc: 83.42%
Epoch 10/20 | Loss: 0.3688 |  Acc: 84.40%
Epoch 11/20 | Loss: 0.3513 |  Acc: 84.77%
Epoch 12/20 | Loss: 0.3342 |  Acc: 86.07%
Epoch 13/20 | Loss: 0.3285 |  Acc: 86.03%
Epoch 14/20 | Loss: 0.3059 |  Acc: 87.06%
Epoch 15/20 | Loss: 0.2948 |  Acc: 87.47%
Epoch 16/20 | Loss: 0.2837 |  Acc: 88.41%
Epoch 17/20 | Loss: 0.2690 |  Acc: 88.62%
Epoch 18/20 | Loss: 0.2716 |  Acc: 88.89%
Epoch 19/20 | Loss: 0.2632 |  Acc: 88.84%
Epoch 20/20 | Loss: 0.2583 |  Acc: 89.32%


In [11]:
torch.save(model.state_dict(), "model.pth")

In [14]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print("Test Accuracy:", test_accuracy)

Test Accuracy: 90.16312407315867
